In [15]:
import torch
from torch.nn import functional as F 
import torch.nn as nn
torch.manual_seed(1337)

In [16]:
B,T,C=4,8,2 #batch, Time, Channels
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [17]:
# Aqui estoy aplicando lo de las notas del Ipad para poder obtener la matriz que calcula los promedios de los 
# vectores anteriores de una cadena de tokens donde cada token es un vector
tril=torch.tril(torch.ones(T,T))
wei=torch.zeros(T,T)
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei, dim=-1)

In [18]:
#bow=bag of words
xbow=wei @ x[0]
print(x[0])
print(xbow)
#Ver como cada fila es el promedio de sus antesesoras, excepto la fila 0 ya que no tiene vectores antes de si 

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [19]:
#para hacer la multiplicacion para todo un batch de tamaño B lo hacemos asi 
# xbow=wei @ x pero de esta manera no concuerdan las dimensiones porque tenemos 
# wei (T,T) y X (B,T,C) necesitamos hacer que wei sea (B,T,T) para que cuadren las dimensiones 
# pero pytorch lo hace por nosotros sin necesidad de aplicar cambios 
xbow=wei @ x

## SELF ATTENTION Q K y V 

In [26]:
head_size=16
B,T,C=4,5,8 #batch, Time, Channels
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 5, 8])

In [27]:
# Capas que generaran los querys keys y values, reciben el input que es la cantidad de embeddings representando cada token y retornan una cantidad de head size
queries=nn.Linear(C,head_size, bias=False)
keys=nn.Linear(C,head_size, bias=False)
values=nn.Linear(C,head_size, bias=False)

In [28]:
Q=queries(x) # (B,T,head_size)
K=keys(x) # (B,T,head_size)

# hacemos el primer poroducto punto de Q y K para obtener la matriz de afinidades entre los tokens
# como debemos hacer coincidir las dimensiones en K tenemos que invertir o transponer las ultimas dos dimensiones, ya que necesitamos multiplicar una matris T*head_size por una head_size*T
wei= Q @ K.transpose(-2,-1)

In [29]:
wei[0]
# Ahora, necesitamos para un decoder la triangular inferior de esta matriz

tensor([[-0.3582,  0.5034, -1.1292, -0.7218,  0.3928],
        [ 0.4390, -0.7248,  1.4080, -0.3444, -1.4142],
        [-0.3618, -0.1769, -0.4395,  0.4482, -0.0451],
        [-1.2591,  0.6253,  0.2372,  2.5060, -0.1820],
        [ 0.7898,  0.4018, -0.8234, -1.6738,  0.7148]],
       grad_fn=<SelectBackward0>)

In [33]:
tril=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(tril==0,float('-inf')) #si quitamos esta linea queda un bloque de encoder
wei[0]

tensor([[-0.3582,    -inf,    -inf,    -inf,    -inf],
        [ 0.4390, -0.7248,    -inf,    -inf,    -inf],
        [-0.3618, -0.1769, -0.4395,    -inf,    -inf],
        [-1.2591,  0.6253,  0.2372,  2.5060,    -inf],
        [ 0.7898,  0.4018, -0.8234, -1.6738,  0.7148]],
       grad_fn=<SelectBackward0>)

In [34]:
# finalmente normalizamos aplicando softmax
wei=F.softmax(wei, dim=-1)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7620, 0.2380, 0.0000, 0.0000, 0.0000],
        [0.3197, 0.3846, 0.2958, 0.0000, 0.0000],
        [0.0181, 0.1192, 0.0809, 0.7818, 0.0000],
        [0.3460, 0.2347, 0.0689, 0.0295, 0.3210]], grad_fn=<SelectBackward0>)

In [36]:
# Luego obtenemos (QK_t)V
xbow=wei @ x